In [ ]:
import requests
import pandas as pd

def fetch_destatis_table(username, password, table_name):
    """
    Holt eine komplette Tabelle von der Destatis GENESIS REST-API.
    """
    # Basis-URL der GENESIS REST-API (Version 2020)
    base_url = "https://www-genesis.destatis.de/genesisWS/rest/2020/data/table"
    
    # Parameter für die Anfrage
    params = {
        "username": username,
        "password": password,
        "name": table_name,      # Der Code der Tabelle (z.B. 12411-0015)
        "area": "all",           # Gesamtes Bundesgebiet
        "compress": "false",     # Wir wollen lesbaren Text, kein ZIP
        "transpose": "false", 
        "language": "de"
    }
    
    print(f"Frage Tabelle {table_name} von GENESIS ab...")
    
    # Anfrage senden
    response = requests.get(base_url, params=params)
    
    # Prüfen, ob der Login und die Anfrage erfolgreich waren
    if response.status_code == 200:
        print("Erfolgreich! Daten werden verarbeitet...")
        
        # Die API gibt einen sehr speziellen String zurück.
        # Wir trennen ihn an den Zeilenumbrüchen auf.
        raw_text = response.text
        lines = raw_text.split('\n')
        
        # ACHTUNG: Die Destatis-CSV-Ausgabe hat oft sehr viele Metadaten/Kopfzeilen am Anfang
        # und Fußnoten am Ende. Für einen schnellen Test geben wir hier erstmal nur 
        # die rohen Zeilen aus, damit du die Struktur siehst.
        
        # Wir wandeln die Zeilen (Semikolon-getrennt) in eine Liste von Listen um
        data_rows = [line.split(';') for line in lines if line.strip()]
        
        # In einen DataFrame packen (Achtung: Dieser DataFrame muss noch stark 
        # bereinigt werden, da die Überschriften bei GENESIS komplex sind)
        df = pd.DataFrame(data_rows)
        return df
        
    elif response.status_code == 401:
        print("Fehler 401: Login fehlgeschlagen. Bitte Username/Passwort prüfen.")
        return None
    else:
        print(f"Fehler {response.status_code}: {response.text}")
        return None

# ==========================================
# TESTLAUF
# ==========================================
if __name__ == "__main__":
    # DEINE ZUGANGSDATEN HIER EINTRAGEN
    # WICHTIG: Wenn du das später auf GitHub lädst, ersetze dies unbedingt durch
    # Umgebungsvariablen (z.B. mit python-dotenv), damit dein Passwort geheim bleibt!
    GENESIS_USER = "DEIN_USERNAME" 
    GENESIS_PASS = "DEIN_PASSWORT"
    
    # Tabelle 12411-0015: "Bevölkerung: Kreise, Stichtag"
    test_df = fetch_destatis_table(GENESIS_USER, GENESIS_PASS, "12411-0015")
    
    if test_df is not None:
        print("\nHier sind die ersten 20 Zeilen der rohen Destatis-Daten:")
        # Wir schauen uns die ersten 20 Zeilen an, weil Destatis oft 
        # einen riesigen Header aus Metadaten vor die eigentliche Tabelle packt.
        display(test_df.head(20))